# 02 - Sources: a fault made of SRF2 subfaults

A `FaultSource` in ShakerMaker is just a container of `PointSource` objects.
Each `PointSource` lives at a position `[x, y, z]` (km, with `z` pointing
down), carries a fault-plane orientation `[strike, dip, rake]` (deg), and is
convolved with a **source time function** (STF).

Here we lay a small **2x5 grid** of subfaults around a hypocenter. Every
subfault uses an `SRF2` slip-rate STF, and we give each one a slightly
**random rake and slip** (fixed seed, so the figure is reproducible). This
mimics how a finite-fault rupture is discretized into many point sources.

We then visualize the geometry with `SourcePlot`, coloring the points by
their slip, and save the figure as a `.png` in this folder.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# SciPy compatibility shim: SourcePlot imports scipy.integrate.trapz,
# which newer SciPy renamed to trapezoid. Alias it before importing.
import scipy.integrate as _si
if not hasattr(_si, "trapz"):
    _si.trapz = _si.trapezoid

from shakermaker.cm_library.LOH import SCEC_LOH_1
from shakermaker.pointsource import PointSource
from shakermaker.faultsource import FaultSource
from shakermaker.stf_extensions.srf2 import SRF2
from shakermaker.tools.plotting import SourcePlot

## Build the subfaults

We start from the LOH.1 crust (context only, not used in construction) and a
hypocenter at 3 km depth. The grid runs 2 columns along strike and 5 rows
down-dip, giving 10 subfaults. A fixed `default_rng(0)` makes the random
rake (base +/- 10 deg) and slip (0.5-1.5 m) reproducible.

In [ ]:
crust = SCEC_LOH_1()                      # layered crust (context)
rng = np.random.default_rng(0)           # fixed seed -> reproducible

x0, y0, z0 = 0.0, 0.0, 3.0               # hypocenter (km)
dx, dz = 0.5, 0.5                        # grid spacing (km)
strike, dip = 0.0, 90.0
base_rake = 0.0

sources = []
for i in range(2):                        # 2 along-strike
    for j in range(5):                    # 5 down-dip -> 10 subfaults
        x = x0 + i * dx
        z = z0 + j * dz                   # deeper down-dip
        rake = base_rake + rng.uniform(-10, 10)     # deg
        slip = rng.uniform(0.5, 1.5)                # m
        stf = SRF2(Tr=2.0, Tp=0.1, Te=1.5, dt=0.01, slip=slip, a=1.0, b=1.0)
        sources.append(PointSource([x, y0, z], [strike, dip, rake], stf=stf))

fault = FaultSource(sources, metadata={"name": "fault_srf2"})
print("nsources =", fault.nsources)

## Visualize the fault geometry

`SourcePlot` draws each subfault as a point in 3-D (Easting, Northing,
Depth). We color by `slip` so the random slip distribution is visible. The
figure is saved as `sources_geometry.png`.

In [ ]:
SourcePlot(fault, show=False, colorby="slip", colorbar=True)
plt.gcf().suptitle("Fault geometry (2x5 SRF2 subfaults)")
plt.gcf().savefig("sources_geometry.png", dpi=150, bbox_inches="tight")